# LCEL 체인을 통한 챗봇 구현


## OpenAI LLM 준비


In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# .env 파일 로드
load_dotenv()

# LLM 준비
llm = init_chat_model(model="openai:gpt-4o", temperature=0.1)

print("✅ LLM 준비 완료")

## 프롬프트 - 시스템 역할 메시지 및 사용자 메시지 구성


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_message = """
당신은 명탐정 코난 영화 전문가입니다. 
사용자가 입력한 연도에 개봉한 명탐정 코난 영화 제목과 등장 인물 및 줄거리를 간단하게 설명하세요.
해당 연도에 개봉한 명탐정 코난 영화가 없는 경우, "해당 연도에 개봉한 명탐정 코난 영화가 없습니다."라고 답변하세요.
"""

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            system_message,
        ),
        (
            "human",
            "{question}",
        ),
    ]
)

print("✅ 프롬프트 템플릿 메시지 구성 완료")

## LCEL 체인 구성

체인 (프롬프트 → LLM → 출력 파서)를 파이프(|)로 연결


In [ ]:
from langchain_core.output_parsers import StrOutputParser

# LCEL 체인 구성 : 프롬프트 + LLM + 출력 파서
chain = prompt | llm | StrOutputParser()

print("✅ LCEL 체인 구성 완료")

ai_message = chain.invoke(
    {"question": "2002년에 개봉한 명탐정 코난 영화는 무엇인가요?"}
)
print(ai_message)

## LLM 응답 메시지의 길이가 궁금할 때 (`RunnableLambda`)

- LLM 응답 메시지의 길이(len) 계산하기.


In [ ]:
# llm 응답 메시지의 길이 계산하여 chain 출력으로 반환
from langchain_core.runnables import RunnableLambda

count_response_length = RunnableLambda(lambda x: len(x))

chain_with_length = chain | count_response_length

response_length = chain_with_length.invoke(
    {"question": "2002년에 개봉한 명탐정 코난 영화는 무엇인가요?"}
)
print(f"LLM 응답 메시지 길이: {response_length}자")

## Gradio UI 구성


In [ ]:
import gradio as gr


# Gradio UI 이벤트 핸들러 함수
def conan_expert_reply(question: str, chat_history: list) -> tuple[list, str]:
    """사용자의 질문과 대화 기록을 받아 명탐정 코난 전문가의 답변을 반환하는 함수"""

    if question and question.strip():
        chat_history += [{"role": "user", "content": question}]
        ai_message = chain.invoke({"question": question})
        return chat_history + [{"role": "assistant", "content": ai_message}], ""
    return chat_history + [{"role": "assistant", "content": "질문을 입력해주세요."}], ""


with gr.Blocks() as demo:
    gr.Markdown("### 🕵️‍♂️ 명탐정 코난 영화 전문가")

    # 챗봇 구성
    chatbot = gr.Chatbot(label="코난 전문가 챗봇", height=200)
    user_input = gr.Textbox(
        label="명탐정 코난 개봉 연도", placeholder="2023년 개봉된 명탐정 코난 영화는?"
    )

    # 입력 이벤트 처리
    user_input.submit(
        fn=conan_expert_reply,
        inputs=[user_input, chatbot],
        outputs=[chatbot, user_input],
    )

# Gradio 실행
demo.launch()

In [ ]:
demo.close()